# Rewards â€” Explore Polymarket's Liquidity Rewards

Polymarket offers **liquidity rewards** to market makers who provide tight quotes on active markets.
The CLOB rewards API has 7 endpoints â€” 3 are public and let you explore reward configurations
without any authentication. The other 4 require L2 HMAC auth and return per-user earnings data.

This notebook covers the 3 public endpoints:

| Method | Description |
|---|---|
| `get_rewards_markets_current` | Currently active reward configs |
| `get_rewards_markets_multi` | Search/filter/sort markets with rewards |
| `get_rewards_market` | Single market reward config by condition ID |

In [ ]:
!pip install -q polymarket-pandas

In [ ]:
from polymarket_pandas import PolymarketPandas
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

client = PolymarketPandas()

## Active Reward Configs

`get_rewards_markets_current` returns all currently active reward configurations.
It uses cursor-based pagination and returns a dict with `data` (DataFrame),
`next_cursor`, `count`, and `limit`.

Setting `expand_rewards_config=True` flattens the nested config list into
prefixed columns like `rewardsConfigAssetAddress` and `rewardsConfigRatePerDay`.

In [ ]:
from polymarket_pandas import PolymarketAPIError

try:
    current = client.get_rewards_markets_current(sponsored=True, expand_rewards_config=True)
    print(f"Markets on this page: {current['count']}")
    print(f"Next cursor:          {current['next_cursor']}")
    print(f"Page limit:           {current['limit']}")
    print()
    current["data"].head(10)
except PolymarketAPIError as e:
    print(f"Server error (Polymarket DB timeout — retry later): {e}")
    current = None


## Markets with Rewards

 supports rich filtering and sorting.
Here we fetch the first 10 markets in descending order.


In [ ]:
try:
    multi = client.get_rewards_markets_multi(
        position="DESC",
        page_size=10,
        expand_rewards_config=True,
    )
    print(f"Markets returned: {multi['count']}")
    display(multi["data"])
except PolymarketAPIError as e:
    print(f"Server error (retry later): {e}")
    multi = None


## Single Market Reward Config

Pick a `conditionId` from the multi-market result and drill into its
full reward configuration, including per-token breakdowns.

In [ ]:
# Pick a conditionId from whichever result is available
cid = None
if multi is not None and not multi["data"].empty:
    cid = multi["data"]["conditionId"].iloc[0]
elif current is not None and not current["data"].empty:
    cid = current["data"]["conditionId"].iloc[0]

if cid:
    print(f"Condition ID: {cid}")
    try:
        single = client.get_rewards_market(
            condition_id=cid,
            expand_rewards_config=True,
            expand_tokens=True,
        )
        print(f"Rows (one per token x config): {single['count']}")
        display(single["data"])
    except PolymarketAPIError as e:
        print(f"Server error: {e}")
else:
    print("No condition IDs available — rewards API may be temporarily down.")


## Private Endpoints (require auth)

The remaining 4 rewards endpoints require **L2 HMAC authentication**
(API key, secret, and passphrase). They return per-user earnings data:

| Method | Description |
|---|---|
| `get_rewards_earnings` | Per-market earnings for a specific day |
| `get_rewards_earnings_total` | Total earnings by asset for a day |
| `get_rewards_percentages` | Real-time reward % per market |
| `get_rewards_user_markets` | Earnings + full market configs (search/filter) |

To use these, create a client with credentials:

```python
client = PolymarketPandas(
    private_key="0x...",
    # Or set env vars: POLYMARKET_API_KEY, POLYMARKET_API_SECRET, POLYMARKET_API_PASSPHRASE
)
client.derive_api_key()  # auto-sets L2 credentials
```

In [ ]:
# Requires POLYMARKET_API_KEY, POLYMARKET_API_SECRET, POLYMARKET_API_PASSPHRASE, POLYMARKET_ADDRESS
# (or call client.derive_api_key() after setting private_key)

# earnings = client.get_rewards_earnings(
#     date="2026-04-10",
#     signature_type=1,
#     maker_address=client.address,
# )
# earnings["data"].head()

# total = client.get_rewards_earnings_total(
#     date="2026-04-10",
#     signature_type=1,
#     maker_address=client.address,
# )
# total

# percentages = client.get_rewards_percentages(
#     signature_type=1,
#     maker_address=client.address,
# )
# percentages  # dict: condition_id -> percentage

# user_markets = client.get_rewards_user_markets(
#     date="2026-04-10",
#     signature_type=1,
#     maker_address=client.address,
#     order_by="earnings",
#     position="DESC",
#     expand_rewards_config=True,
#     expand_tokens=True,
#     expand_earnings=True,
# )
# user_markets["data"].head()